# Requirement 3: Tamper resistance

> Administrator changes to the execution environment must either preserve confidentiality or cause the application to reject the environment before releasing sensitive information.

**Covered by:** Both

**How it is fulfilled:** Prevention: administrators have no data-plane rights, so they cannot change a running sandbox. Detection: before releasing data the orchestrator reads the configuration back and fails closed on any difference.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
from helpers import *
sandbox, session_id = new_session()

In [3]:
results = verify_session(sandbox)
for name, ok in results.items():
    check(name, ok)
print("release data?", all(results.values()))

PASS  disk image is the approved one
PASS  created from a disk, not a snapshot
PASS  egress default is Deny
PASS  no ports exposed
PASS  suspend disabled
PASS  auto-delete enabled
release data? True


In [4]:
# Simulate tampering: loosen egress after the session was approved
sandbox.set_egress_default("Allow")
results = verify_session(sandbox)
check("read-back detects the change", not results["egress default is Deny"])
print("release data?", all(results.values()), "-> the orchestrator fails closed and destroys the session")

PASS  read-back detects the change
release data? False -> the orchestrator fails closed and destroys the session


In [5]:
cleanup(sandbox)

**Caveat:** Only a data-plane holder could make this change in practice, so the demo performs it as the orchestrator to show the detection. Control-plane policy and locks add a further layer.